# BEST_SOLUTION — strong memory-aware pipeline

Улучшенный пайплайн для задачи «Потеряшки»:
- валидация с маскированием 20% позитивов в incident-окне;
- расширенная генерация кандидатов (ALS + ItemKNN + co-visitation + popularity + TF-IDF content);
- расширенный набор фичей (user/item/source/history/content);
- ансамбль двух CatBoostRanker моделей;
- контроль памяти (`float32`, `category`, батчевые операции).


In [ ]:
# !pip install implicit catboost scikit-learn -q
import os, gc, warnings, math
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from catboost import CatBoostRanker, Pool
import implicit
from tqdm.auto import tqdm

SEED = 42
TOP_K = 20
CANDS_PER_USER = 250
ALS_FACTORS = 128
ALS_ITERS = 35
KNN_K = 200
VAL_HIDDEN_SHARE = 0.2

DATA_DIR = './data'  # поменяй при необходимости


In [ ]:
def reduce_mem(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast='integer')
        elif pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast='float')
    return df

interactions = pd.read_csv(f"{DATA_DIR}/interactions.csv", parse_dates=['event_ts'])
targets = pd.read_csv(f"{DATA_DIR}/targets.csv")
editions = pd.read_csv(f"{DATA_DIR}/editions.csv")
book_genres = pd.read_csv(f"{DATA_DIR}/book_genres.csv")
users = pd.read_csv(f"{DATA_DIR}/users.csv")

for df_name in ['interactions','targets','editions','book_genres','users']:
    globals()[df_name] = reduce_mem(globals()[df_name])

interactions = interactions[interactions.event_type.isin([1,2])].copy()
interactions['w'] = np.where(interactions.event_type.eq(2), 1.4, 1.0).astype('float32')

print(interactions.shape, targets.shape, editions.shape)


In [ ]:
def make_validation_split(interactions: pd.DataFrame, share=VAL_HIDDEN_SHARE, seed=SEED):
    rng = np.random.default_rng(seed)
    t_max = interactions.event_ts.max()
    inc_start = t_max - pd.Timedelta(days=31)
    inc = interactions[interactions.event_ts >= inc_start].copy()
    hist = interactions[interactions.event_ts < inc_start].copy()

    pos_pairs = inc[['user_id','edition_id']].drop_duplicates()
    mask_idx = []
    for u, g in pos_pairs.groupby('user_id').groups.items():
        idx = np.array(list(g))
        m = max(1, int(len(idx) * share))
        mask_idx.extend(rng.choice(idx, size=min(m, len(idx)), replace=False))
    hidden_pairs = pos_pairs.loc[mask_idx].copy()
    hidden_pairs['target'] = 1

    hidden_set = set(map(tuple, hidden_pairs[['user_id','edition_id']].to_numpy()))
    keep = inc[~inc[['user_id','edition_id']].apply(tuple, axis=1).isin(hidden_set)]
    train_obs = pd.concat([hist, keep], ignore_index=True)
    return train_obs, hidden_pairs

def ndcg_at_k(pred: pd.DataFrame, gt: pd.DataFrame, k=20):
    gt_map = gt.groupby('user_id')['edition_id'].apply(set).to_dict()
    scores = []
    for u, g in pred.groupby('user_id'):
        rel = np.array([1.0 if e in gt_map.get(u, set()) else 0.0 for e in g.sort_values('rank').edition_id.values[:k]])
        if rel.sum() == 0:
            scores.append(0.0)
            continue
        dcg = (rel / np.log2(np.arange(2, len(rel)+2))).sum()
        ideal = np.sort(rel)[::-1]
        idcg = (ideal / np.log2(np.arange(2, len(ideal)+2))).sum()
        scores.append(float(dcg / (idcg + 1e-12)))
    return float(np.mean(scores))


In [ ]:
def fit_index(obs: pd.DataFrame):
    users = pd.Index(obs.user_id.unique(), name='user_id')
    items = pd.Index(editions.edition_id.unique(), name='edition_id')
    u2i = {u:i for i,u in enumerate(users)}
    e2i = {e:i for i,e in enumerate(items)}
    return users, items, u2i, e2i

def build_ui(obs: pd.DataFrame, u2i, e2i):
    x = obs[obs.user_id.isin(u2i) & obs.edition_id.isin(e2i)]
    r = x.user_id.map(u2i).to_numpy()
    c = x.edition_id.map(e2i).to_numpy()
    w = x.w.astype('float32').to_numpy()
    ui = sparse.csr_matrix((w,(r,c)), shape=(len(u2i), len(e2i)), dtype='float32')
    return ui

def train_retrievers(ui):
    als = implicit.als.AlternatingLeastSquares(
        factors=ALS_FACTORS, iterations=ALS_ITERS, regularization=0.01,
        random_state=SEED, calculate_training_loss=False
    )
    als.fit((ui.T * 20).astype('float32'))

    knn = implicit.nearest_neighbours.CosineRecommender(K=KNN_K)
    knn.fit(ui.T)
    return als, knn


In [ ]:
def build_text_embeddings(editions: pd.DataFrame):
    txt = (editions.title.fillna('') + ' ' + editions.description.fillna('')).str.lower()
    tfidf = TfidfVectorizer(max_features=120000, ngram_range=(1,2), min_df=3)
    X = tfidf.fit_transform(txt)
    svd = TruncatedSVD(n_components=128, random_state=SEED)
    emb = normalize(svd.fit_transform(X).astype('float32'))
    emb_map = pd.DataFrame({'edition_id': editions.edition_id.values})
    for i in range(emb.shape[1]):
        emb_map[f'txt_{i}'] = emb[:, i]
    return emb_map.set_index('edition_id'), tfidf, svd

def user_text_profile(obs, txt_item_emb):
    tmp = obs[['user_id','edition_id','w']].merge(txt_item_emb.reset_index(), on='edition_id', how='left').fillna(0)
    txt_cols = [c for c in tmp.columns if c.startswith('txt_')]
    for c in txt_cols:
        tmp[c] = tmp[c] * tmp['w']
    prof = tmp.groupby('user_id')[txt_cols].mean().astype('float32')
    return prof


In [ ]:
def gen_candidates_for_users(user_list, obs, users_idx, items_idx, u2i, e2i, als, knn, txt_item_emb, txt_user_emb, cands_n=CANDS_PER_USER):
    pop = obs.groupby('edition_id').size().sort_values(ascending=False).head(500).index.to_numpy()
    user_hist = obs.groupby('user_id')['edition_id'].agg(set).to_dict()
    i2e = np.array(items_idx)
    ui = build_ui(obs, u2i, e2i)

    rows = []
    mat = txt_item_emb.to_numpy(dtype='float32')
    for u in tqdm(user_list):
        seen = user_hist.get(u, set())
        cand = {}

        if u in u2i:
            uid = u2i[u]
            rec, score = als.recommend(uid, ui[uid], N=180, filter_already_liked_items=False)
            for iid, s in zip(rec, score):
                e = int(i2e[iid]); cand[e] = max(cand.get(e, -1e9), float(s))

            item_ids = list(seen)[:30]
            for e in item_ids:
                if e in e2i:
                    sim_i, sim_s = knn.similar_items(e2i[e], N=40)
                    for iid, s in zip(sim_i, sim_s):
                        ee = int(i2e[iid]); cand[ee] = max(cand.get(ee, -1e9), float(s)*0.7)

        if u in txt_user_emb.index:
            uv = txt_user_emb.loc[u].to_numpy(dtype='float32')
            s = mat @ uv
            top = np.argpartition(s, -120)[-120:]
            for j in top:
                e = int(txt_item_emb.index[j]); cand[e] = max(cand.get(e, -1e9), float(s[j])*0.9)

        for e in pop[:100]:
            cand[int(e)] = max(cand.get(int(e), -1e9), 0.05)

        cand = [(e,s) for e,s in cand.items() if e not in seen]
        cand = sorted(cand, key=lambda x: x[1], reverse=True)[:cands_n]
        for r,(e,s) in enumerate(cand,1):
            rows.append((u,e,r,s))

    out = pd.DataFrame(rows, columns=['user_id','edition_id','cand_rank','cand_score'])
    return out


In [ ]:
def build_features(cands, obs, editions, users, book_genres, als, u2i, e2i):
    item_pop = obs.groupby('edition_id').size().rename('item_pop').astype('float32')
    user_pop = obs.groupby('user_id').size().rename('user_pop').astype('float32')

    ug = (obs[['user_id','edition_id']]
          .merge(editions[['edition_id','book_id']], on='edition_id', how='left')
          .merge(book_genres, on='book_id', how='left')
          .dropna())
    ug_cnt = ug.groupby(['user_id','genre_id']).size().rename('ug_cnt').reset_index()
    bg = editions[['edition_id','book_id']].merge(book_genres, on='book_id', how='left')

    df = cands.merge(editions[['edition_id','author_id','publication_year','language_id','publisher_id']], on='edition_id', how='left')
    df = df.merge(users[['user_id','gender','age']], on='user_id', how='left')
    df = df.merge(item_pop, on='edition_id', how='left').merge(user_pop, on='user_id', how='left')
    df['item_pop'] = df['item_pop'].fillna(0)
    df['user_pop'] = df['user_pop'].fillna(0)

    df = df.merge(bg[['edition_id','genre_id']], on='edition_id', how='left')
    df = df.merge(ug_cnt, on=['user_id','genre_id'], how='left')
    df['ug_cnt'] = df['ug_cnt'].fillna(0)

    als_u = np.zeros((len(df), ALS_FACTORS), dtype='float32')
    als_i = np.zeros((len(df), ALS_FACTORS), dtype='float32')
    uok = df.user_id.map(u2i).fillna(-1).astype(int).to_numpy()
    iok = df.edition_id.map(e2i).fillna(-1).astype(int).to_numpy()
    good = (uok >= 0) & (iok >= 0)
    als_u[good] = als.user_factors[uok[good]]
    als_i[good] = als.item_factors[iok[good]]
    df['als_dot'] = (als_u * als_i).sum(1)
    df['als_l2'] = np.linalg.norm(als_u-als_i, axis=1)

    for c in ['gender','language_id','publisher_id','author_id']:
        df[c] = df[c].fillna(-1).astype('int32')
    for c in ['age','publication_year','cand_rank','item_pop','user_pop','ug_cnt','cand_score','als_dot','als_l2']:
        df[c] = df[c].fillna(0).astype('float32')

    return df


In [ ]:
def train_ranker(train_df, y):
    train_df = train_df.sort_values('user_id').reset_index(drop=True)
    y = y[train_df.index]
    cat_cols = ['gender','language_id','publisher_id','author_id']
    feat_cols = [c for c in train_df.columns if c not in ['user_id','edition_id']]

    pool = Pool(train_df[feat_cols], label=y, group_id=train_df['user_id'], cat_features=[feat_cols.index(c) for c in cat_cols if c in feat_cols])

    m1 = CatBoostRanker(
        loss_function='YetiRankPairwise',
        iterations=1200, learning_rate=0.05, depth=8,
        random_seed=SEED, verbose=200
    )
    m2 = CatBoostRanker(
        loss_function='PairLogitPairwise',
        iterations=1000, learning_rate=0.06, depth=7,
        random_seed=SEED+13, verbose=200
    )
    m1.fit(pool)
    m2.fit(pool)
    return (m1, m2), feat_cols

def infer_rank(cands_df, models, feat_cols, topk=20):
    m1, m2 = models
    s1 = m1.predict(cands_df[feat_cols])
    s2 = m2.predict(cands_df[feat_cols])
    cands_df = cands_df.copy()
    cands_df['score'] = 0.6*s1 + 0.4*s2
    cands_df = cands_df.sort_values(['user_id','score'], ascending=[True,False])
    cands_df['rank'] = cands_df.groupby('user_id').cumcount() + 1
    return cands_df[cands_df['rank']<=topk][['user_id','edition_id','rank']]


In [ ]:
# ===== Validation =====
obs_train, hidden_gt = make_validation_split(interactions)
users_idx, items_idx, u2i, e2i = fit_index(obs_train)
ui = build_ui(obs_train, u2i, e2i)
als, knn = train_retrievers(ui)

txt_item_emb, tfidf, svd = build_text_embeddings(editions)
txt_user_emb = user_text_profile(obs_train, txt_item_emb)

val_users = hidden_gt.user_id.unique()
val_cands = gen_candidates_for_users(val_users, obs_train, users_idx, items_idx, u2i, e2i, als, knn, txt_item_emb, txt_user_emb)
val_pairs = hidden_gt[['user_id','edition_id']].drop_duplicates().assign(target=1)
train_rank_df = build_features(val_cands, obs_train, editions, users, book_genres, als, u2i, e2i)
train_rank_df = train_rank_df.merge(val_pairs, on=['user_id','edition_id'], how='left')
train_rank_df['target'] = train_rank_df['target'].fillna(0).astype('int8')

models, feat_cols = train_ranker(train_rank_df.drop(columns=['target']), train_rank_df['target'].to_numpy())
val_pred = infer_rank(train_rank_df.drop(columns=['target']), models, feat_cols, topk=20)
print('VAL NDCG@20 =', ndcg_at_k(val_pred, hidden_gt[['user_id','edition_id']]))


In [ ]:
# ===== Full train + submission =====
full_obs = interactions.copy()
users_idx, items_idx, u2i, e2i = fit_index(full_obs)
ui = build_ui(full_obs, u2i, e2i)
als, knn = train_retrievers(ui)

txt_item_emb, tfidf, svd = build_text_embeddings(editions)
txt_user_emb = user_text_profile(full_obs, txt_item_emb)

test_users = targets.user_id.unique()
test_cands = gen_candidates_for_users(test_users, full_obs, users_idx, items_idx, u2i, e2i, als, knn, txt_item_emb, txt_user_emb)

test_rank_df = build_features(test_cands, full_obs, editions, users, book_genres, als, u2i, e2i)
sub = infer_rank(test_rank_df, models, feat_cols, topk=20)
sub.to_csv('submission.csv', index=False)
sub.head()
